# 05 · XLM-R long-text policy: head-only versus head plus tail

**Question.** XLM-R reads at most 512 tokens, and in notebook 03 its macro-F1 on documents over 1,024 words fell far below the TF-IDF baseline. Does keeping the end of a long document, instead of only its beginning, recover part of that loss?

**Policies.** With two special tokens the 512-token budget leaves 510 content tokens. Head-only keeps the first 510 tokens. Head plus tail keeps the first 255 and the last 255. Documents that fit the budget are identical under both policies, so any difference comes from documents longer than 510 tokens. No chunk receives a document label.

**Controlled design.** Both runs fine-tune `FacebookAI/xlm-roberta-base` with the recipe of notebook 03 on the same 40,001 training documents, select the checkpoint by macro-F1 on the same 2,001 development documents and are evaluated once on the full fixed holdout of 54,471 documents. The only difference is the truncation policy; the stage refuses runs whose configuration, tokenizer, label order or source code differ otherwise. Both runs use the corrected loss handling (`accepts_loss_kwargs=False`), whereas the original head-only run of notebook 03 divided the document loss by the accumulation factor. That original run is kept unchanged and shown for reference only.

**Criteria.** Fixed before training and identical to the other experiments: head plus tail is adopted over head-only if the paired holdout macro-F1 gain is at least 0.003 with a family-cluster bootstrap 95% interval above zero, the family-clean subset is not worse and human F1 drops by at most 0.003. The policy with the higher development macro-F1 is also recorded, because selection between policies should not depend on the holdout. This notebook restores saved results and trains nothing.

## 01 · Verified runs

Training ran on Kaggle T4 GPUs through `experiments/02_xlm_roberta_kaggle.ipynb` with `MODE = "train"`. Only the evaluation outputs, training history and selected-model manifests were downloaded, not the weights. The `xlmr-long-text` stage of `scripts/run_rewrite_experiments.py` checks each holdout receipt against the saved predictions, metrics and selected-model manifest, aligns predictions to the fixed holdout IDs and evaluates them with the same family-cluster bootstrap and family-clean subset as notebook 04.

In [1]:
import json
import sys
import tomllib
from pathlib import Path

import plotly.express as px
import plotly.io
import polars as pl
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from hm3.validation import LABELS, fingerprint_file

plotly.io.renderers.default = "plotly_mimetype"
LONG_TEXT = tomllib.loads((ROOT / "config/xlmr_long_text.toml").read_text())
XLMR = tomllib.loads((ROOT / "config/xlmr.toml").read_text())
STAGE = ROOT / "runs/rewrite_experiments/xlmr_long_text"
record = json.loads((STAGE / "record.json").read_text())
for name, expected in record["files"].items():
    if fingerprint_file(STAGE / name) != expected:
        raise ValueError(f"Changed result file: {name}")
metrics = json.loads((STAGE / "metrics.json").read_text())
display(
    {key: value for key, value in record.items() if key != "files"},
    pl.DataFrame(
        [{"experiment": name, **settings} for name, settings in XLMR["experiments"].items()]
    ),
)

{'identity': '66b4203aa37339e67f6eb834c861c97ee0c8feb64b97234358615052016da48c',
 'peak_rss_gib': 0.3206329345703125,
 'stage': 'xlmr-long-text',
 'seconds': 4.563867292003124}

experiment,truncation_policy
str,str
"""xlmr_head_only_40k""","""head"""
"""xlmr_head_tail_40k""","""head_tail"""


In [2]:
RUN_NAMES = {
    "original_head_only": "XLM-R head-only (original accumulation)",
    "head_only": "XLM-R head-only",
    "head_tail": "XLM-R head plus tail",
}
display(
    pl.DataFrame(
        [
            {
                "run": RUN_NAMES[name],
                "kaggle_kernel": LONG_TEXT[name]["kaggle_kernel"],
                "truncation_policy": run["identity"]["configuration"]["truncation_policy"],
                "best_development_macro_f1": run["best_development_macro_f1"],
                "selected_checkpoint": run["best_model_checkpoint"],
                "stopped_at_step": run["global_step"],
                "stopped_at_epoch": run["epoch"],
                "train_runtime_minutes": float(run["train_runtime_seconds"]) / 60,
            }
            for name, run in metrics["runs"].items()
        ]
    ),
    {
        "development_selected_policy": metrics["development_selected_policy"],
        "same_environment": metrics["same_environment"],
        "environment": metrics["runs"]["head_tail"]["identity"]["environment"],
    },
)
histories = []
for name in ["head_only", "head_tail"]:
    run_directory = ROOT / metrics["runs"][name]["run_directory"]
    events = json.loads((run_directory / "history.json").read_text())["events"]
    history = pl.from_dicts(events, infer_schema_length=None)
    if "eval_macro_f1" in history.columns:
        histories.append(
            history.select("step", "eval_macro_f1")
            .drop_nulls()
            .with_columns(pl.lit(RUN_NAMES[name]).alias("run"))
        )
display(
    px.line(
        pl.concat(histories),
        x="step",
        y="eval_macro_f1",
        color="run",
        markers=True,
        title="Development macro-F1 during fine-tuning",
        labels={
            "step": "Optimizer step",
            "eval_macro_f1": "Development macro-F1",
            "run": "Run",
        },
    )
)

run,kaggle_kernel,truncation_policy,best_development_macro_f1,selected_checkpoint,stopped_at_step,stopped_at_epoch,train_runtime_minutes
str,str,str,f64,str,i64,f64,f64
"""XLM-R head-only (original accu…","""ke1rro/notebookf475a1574b""","""head""",0.819411,"""checkpoint-500""",1000,0.79992,24.109467
"""XLM-R head-only""","""ke1rro/hw3-xlmr-head-only-corr…","""head""",0.821172,"""checkpoint-500""",1000,0.79992,24.722828
"""XLM-R head plus tail""","""ke1rro/hw3-xlmr-head-tail-corr…","""head_tail""",0.916404,"""checkpoint-2500""",2502,2.0,63.250393


{'development_selected_policy': 'head_tail',
 'same_environment': True,
 'environment': {'python': '3.13.16',
  'torch': '2.11.0+cu130',
  'cuda': '13.0',
  'gpu': 'Tesla T4',
  'transformers': '5.24.0',
  'accelerate': '1.16.0',
  'tokenizers': '0.23.3',
  'numpy': '2.5.5',
  'polars': '2.1.0',
  'scikit-learn': '1.9.1'}}

## 02 · Fixed-holdout comparison

All rows use the same 54,471 documents. Paired differences resample all-training document families with 1,000 repetitions. The TF-IDF row is the reference refit of notebook 04.

In [3]:
EVALUATION_NAMES = {
    "word_char_tfidf_reference": "Word and character TF-IDF with LR",
    "xlmr_head_only_original_accumulation": "XLM-R head-only (original accumulation)",
    "xlmr_head_only": "XLM-R head-only",
    "xlmr_head_tail": "XLM-R head plus tail",
}


def summary_row(name: str, evaluation: dict) -> dict:
    """Flatten aggregate and per-class F1 for one evaluated model."""
    per_class = {row["label"]: row for row in evaluation["holdout"]["per_class"]}
    return {
        "model": EVALUATION_NAMES[name],
        "holdout_macro_f1": evaluation["holdout"]["macro_f1"],
        "family_clean_macro_f1": evaluation["family_clean"]["macro_f1"],
        **{f"{label}_f1": per_class[label]["f1"] for label in LABELS},
        "fusion_recall": per_class["fusion"]["recall"],
    }


evaluations = metrics["evaluations"]
display(pl.DataFrame([summary_row(name, evaluations[name]) for name in EVALUATION_NAMES]))
display(
    pl.DataFrame(
        [
            {
                "comparison": comparison,
                "holdout_difference": evaluations[name]["paired_holdout"]["difference"],
                "ci95_low": evaluations[name]["paired_holdout"]["ci95_low"],
                "ci95_high": evaluations[name]["paired_holdout"]["ci95_high"],
                "family_clean_difference": evaluations[name]["paired_family_clean"]["difference"],
            }
            for name, comparison in [
                ("xlmr_head_tail", "head plus tail versus head-only"),
                ("xlmr_head_only", "corrected versus original head-only"),
                ("xlmr_head_tail_versus_tfidf", "head plus tail versus TF-IDF reference"),
            ]
        ]
    ),
    {"success": metrics["success"]},
)
for name in ["xlmr_head_only", "xlmr_head_tail"]:
    display(
        px.imshow(
            evaluations[name]["holdout"]["row_normalized_confusion_matrix"],
            x=list(LABELS),
            y=list(LABELS),
            text_auto=".3f",
            title=f"{EVALUATION_NAMES[name]}: fixed-holdout confusion proportions",
            labels={"x": "Predicted class", "y": "True class", "color": "Row proportion"},
        )
    )

model,holdout_macro_f1,family_clean_macro_f1,human_f1,ai_f1,fusion_f1,fusion_recall
str,f64,f64,f64,f64,f64,f64
"""Word and character TF-IDF with…",0.895743,0.893689,0.841125,0.989446,0.856657,0.827817
"""XLM-R head-only (original accu…",0.813427,0.804441,0.74426,0.976578,0.719444,0.640399
"""XLM-R head-only""",0.811687,0.802511,0.750312,0.976945,0.707805,0.608197
"""XLM-R head plus tail""",0.903012,0.900528,0.843107,0.991087,0.874842,0.89013


comparison,holdout_difference,ci95_low,ci95_high,family_clean_difference
str,f64,f64,f64,f64
"""head plus tail versus head-onl…",0.091325,0.087363,0.095123,0.098017
"""corrected versus original head…",-0.00174,-0.003951,0.00051,-0.00193
"""head plus tail versus TF-IDF r…",0.007269,0.004065,0.010656,0.006839


{'success': {'minimum_gain': True,
  'positive_interval': True,
  'family_clean_not_worse': True,
  'human_f1_drop_within_limit': True,
  'passed': True}}

## 03 · Document length

The policies can differ only on documents above the token budget, so the length groups show where any change comes from. Groups use word counts, not tokenizer tokens.

In [4]:
LENGTH_ORDER = ["001-128", "129-512", "513-1024", "1025+"]
lengths = pl.read_parquet(STAGE / "length_bucket_macro_f1.parquet").with_columns(
    pl.col("model").replace(
        {
            "word_char_tfidf_reference": EVALUATION_NAMES["word_char_tfidf_reference"],
            "original_head_only": RUN_NAMES["original_head_only"],
            "head_only": RUN_NAMES["head_only"],
            "head_tail": RUN_NAMES["head_tail"],
        }
    )
)
display(
    lengths.pivot(on="model", index=["length_bucket", "support"], values="macro_f1")
    .with_columns(
        (pl.col(RUN_NAMES["head_tail"]) - pl.col(RUN_NAMES["head_only"])).alias(
            "head_tail_minus_head_only"
        )
    )
    .sort(pl.col("length_bucket").replace_strict(LENGTH_ORDER, list(range(4))))
)
display(
    px.bar(
        lengths,
        x="length_bucket",
        y="macro_f1",
        color="model",
        barmode="group",
        category_orders={"length_bucket": LENGTH_ORDER},
        title="Fixed-holdout macro-F1 by document length",
        labels={
            "length_bucket": "Document length in words",
            "macro_f1": "Macro-F1",
            "model": "Model",
        },
    )
)

length_bucket,support,Word and character TF-IDF with LR,XLM-R head-only (original accumulation),XLM-R head-only,XLM-R head plus tail,head_tail_minus_head_only
str,i64,f64,f64,f64,f64,f64
"""001-128""",5520,0.800403,0.740461,0.741448,0.863163,0.121715
"""129-512""",25240,0.888186,0.86576,0.861365,0.921463,0.060098
"""513-1024""",19013,0.928106,0.790323,0.790697,0.91379,0.123093
"""1025+""",4698,0.912023,0.709682,0.705679,0.832522,0.126843


## 04 · Observations

- Head plus tail reaches holdout macro-F1 0.9030 against 0.8117 for head-only, a paired gain of 0.0913 with 95% interval [0.0874, 0.0951]; the family-clean gain is 0.0980. Human F1 rises from 0.750 to 0.843 and fusion recall from 0.608 to 0.890, so every predeclared criterion is met. Development macro-F1 selects the same policy (0.9164 against 0.8212).
- The two runs did not train for the same number of steps. Head-only stopped early at step 1,000 (0.8 epochs), because development macro-F1 at steps 750 and 1,000 stayed below its step 500 value. Head plus tail also dropped at step 1,000 (0.7538) but improved again at step 1,250, so it used the full budget of two epochs and selected step 2,500.
- Documents with at most 128 words fit the token budget and receive identical inputs under both policies, yet head plus tail gains 0.122 macro-F1 there, about as much as on documents over 1,024 words (0.127). Most of the gain therefore comes from the longer training of the head plus tail run, not from the tail tokens, and this comparison cannot isolate the effect of the policy.
- The confusion matrices show the change: head-only labels 35.8% of fusion documents as human, head plus tail 9.4%, while the share of human documents labelled fusion grows from 11.5% to 17.7%.
- Corrected loss scaling changes the head-only result by $-0.0017$ [$-0.0040$, $0.0005$], within noise; both head-only runs stopped at the same step and selected the same checkpoint.
- Against the TF-IDF reference, head plus tail is better on the full holdout by 0.0073 [0.0041, 0.0107] and on documents up to 512 words, but worse on documents over 1,024 words (0.833 against 0.912) and slightly worse between 513 and 1,024 words.

## 05 · Conclusions and limitations

By the predeclared rule head plus tail replaces head-only as the document-level XLM-R policy: it is better on the holdout, on family-clean documents and on development data. I do not attribute the gain to the tail tokens. Early stopping ended the head-only run after 0.8 epochs, and the gain on short documents with identical inputs is as large as on long ones, so training length decided most of this comparison. Isolating the policy needs both runs trained for the same number of steps, for example two epochs without early stopping, which I did not run.

The document-level encoder now beats the TF-IDF reference by 0.0073, but stays below the TF-IDF model with sentence rewrite scores (0.9236 in notebook 04) and still loses to TF-IDF on documents over 1,024 words. I did not refit it on all training data. The 40k-document model scores 0.90166 on the Kaggle public leaderboard; public scores are not used for any selection.

**Limitations.** Each policy has one seed, GPU training is not bit-for-bit deterministic, and the bootstrap interval covers holdout sampling only, not training variance. Early stopping with patience 2 makes the stopping step sensitive to single noisy development evaluations, such as the drop to 0.7538 at step 1,000 of the head plus tail run.